In [1]:
!pip3 install --upgrade tensorflow
!pip3 install --upgrade xgboost
!pip3 install --upgrade keras
!pip3 install --upgrade scikit-learn
!pip3 install --upgrade keras-tuner
# Restart the notebook after doing this

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.9/644.9 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 71.9 MB/s eta 0:00:00
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.4.1
    Uninstalling ml-dtypes-0.4.1:
      Successfully uninstalled ml-dtypes-0.4.1
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.18.0
    Uninstalling tensorboard-2.18.0:
      Successfully uninstalled tensorboard-2.18.0
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.18.0
    Uninstalling tensorflow-2.18.0:
      Successfully uninstalled tensorflow-2.18.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-decision-forests 1.11.0 requires tensorflow==2.18.0, but you have tens

In [2]:
# Import Needed Libraries
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from skopt import BayesSearchCV
from skopt.space import Real, Integer
from xgboost import XGBRegressor
import random
import keras
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error
import math
import keras_tuner

2025-04-24 01:38:28.720529: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745458708.748933      13 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745458708.757122      13 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1745458708.781321      13 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745458708.781358      13 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745458708.781361      13 computation_placer.cc:177] computation placer alr

In [3]:
# Import Data
train_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
test_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")

In [4]:
# output the predictions in the Kaggle competition format
def write_preds_to_file(file, preds, test, *, is_nn):
    with open(file, "w") as f:
        f.write(f"Id,SalePrice\n")
        for i in range(len(test.index)):
            if is_nn:
                f.write(f'{test.iloc[i].astype("int").get("Id")},{preds[i][0]}\n')
            else:
                f.write(f'{test.iloc[i].astype("int").get("Id")},{preds[i]}\n')



# Missing Data Analysis
Removing features for which more than 6% of the data is missing that feature

In [5]:
# Analyze Missing data
def analyze_missing_data(train):
    num_missing_vals = train.isnull().sum().sort_values(ascending = False)
    missing_percent = (train.isnull().sum()/train.isnull().count()).sort_values(ascending = False)
    missing_data = pd.concat([num_missing_vals, missing_percent], axis=1, keys=['Total', 'Percent'])
    return missing_data
#missing_data.head(20)

In [6]:
# Remove Features with LOTS OF Missing Data
def remove_missing_features(train, test):
    missing_data = analyze_missing_data(train)
    more_than_six_percent_missing = missing_data['Percent'] > 0.06
    features_to_remove = missing_data.loc[more_than_six_percent_missing].index
    train = train.drop(features_to_remove, axis=1, errors='ignore')
    test = test.drop(features_to_remove, axis=1, errors = 'ignore')
    return train, test

train_df, test_df = remove_missing_features(train_df, test_df)

# INSERT CYCLIC ENCODING FOR MONTHS IN THIS SECTION AND ORDINAL ENCODINGS HERE

These are the encodings we should do for ALL models

In [7]:
# DO CYCLIC ENCODING IN THIS CODE BLOCK
def cyclic_encoding(df):
    if 'MoSold' not in df.columns.tolist():
        return df
    df = df.copy()
    df['MoSold'] = df['MoSold'] - 1
    as_rads = df['MoSold'] * (2 * math.pi) / 12
    df['MoSold_sin'] = np.sin(as_rads)
    df['MoSold_cos'] = np.cos(as_rads)
    df.drop('MoSold', axis=1, inplace=True)
    return df

train_df = cyclic_encoding(train_df)
test_df = cyclic_encoding(test_df)

# Feature List by Category

In [8]:
# List of Features: Ordinal features, Nominal features, numerical features
# For now, let's not cut features. If the model training ends up too slow, we can think about cutting features.

ordinal_features = [  'LotShape'
                    , 'LandSlope' # probably can cut
                    , 'ExterQual'
                    , 'BsmtQual'
                    , 'BsmtCond'
                    , 'BsmtExposure'
                    , 'BsmtFinType1'
                    , 'BsmtFinType2'
                    , 'ExterCond'
                    , 'HeatingQC' # probably can cut
                    , 'CentralAir'
                    , 'Electrical'
                    , 'KitchenQual'
                    , 'Functional'
                    , 'GarageType'
                    , 'GarageFinish'
                    , 'GarageQual'
                    , 'GarageCond'
                    , 'PavedDrive'
                   ]
nominal_features = ['MSZoning' # 8 options => # Turn into RL, RM, FV, other => 4 options
                    , 'Street' # 2 options # probably can cut
                    , 'LandContour' # 4 options
                    , 'Utilities' # 4 options
                    , 'LotConfig' # 5 options
                    , 'Neighborhood' # 25 options
                    , 'Condition1' # 9 options # probably can cut
                    , 'Condition2' # 9 options # probably can cut
                    , 'BldgType' # 5 options
                    , 'HouseStyle' # 8 options # Turn into 2Story, 1Story, 1.5Fin, SLvl, other => 5 options
                    , 'RoofStyle' # 6 options # probably can cut
                    , 'RoofMatl' # 8 options # probably can cut
                    , 'Exterior1st' # 17 options # Turn into VinylSd, MetalSd, Wd Sdng, HdBoard, Plywood, other => 6 options
                    , 'Exterior2nd' # 17 options # Turn into VinylSd, MetalSd, HdBoard, Plywood, WdSdng, other => 6 options
                    , 'Foundation' # 6 options
                    , 'Heating' # 6 options # probably can cut
                    , 'SaleType' # 10 options # Turn into WD, New, COD, other => 4 options
                    , 'SaleCondition' # 6 options
                   ]
numerical_features = [  'MSSubClass'
                      , 'LotArea'
                      , 'OverallQual'
                      , 'OverallCond'
                      , 'YearBuilt'
                      , 'YearRemodAdd'
                      , 'MasVnrArea'
                      , 'BsmtFinSF1'
                      , 'BsmtFinSF2'
                      , 'BsmtUnfSF'
                      , 'TotalBsmtSF'
                      , '1stFlrSF'
                      , '2ndFlrSF'
                      , 'LowQualFinSF' # probably can cut
                      , 'GrLivArea'
                      , 'BsmtFullBath'
                      , 'BsmtHalfBath' # probably can cut
                      , 'FullBath'
                      , 'HalfBath'
                      , 'BedroomAbvGr'
                      , 'KitchenAbvGr'
                      , 'TotRmsAbvGrd'
                      , 'Fireplaces'
                      , 'GarageYrBlt'
                      , 'GarageCars'
                      , 'GarageArea'
                      , 'WoodDeckSF'
                      , 'OpenPorchSF'
                      , 'EnclosedPorch' # probably can cut
                      , '3SsnPorch' # probably can cut
                      , 'ScreenPorch' # probably can cut
                      , 'PoolArea' # probably can cut
                      , 'MiscVal' # probably can cut
                      , 'MoSold_sin'
                      , 'MoSold_cos'
                      , 'YrSold'
                    ]

input_features = ordinal_features + nominal_features + numerical_features
output_features = ['SalePrice', 'SalePriceTransformed']

# INSERT ORDINAL ENCODING HERE. The encodings here should apply for ALL Models
TODO Implement Ordinal Encoding. Right now we just discard ordinal features

In [9]:
# DO ORDINAL ENCODINGS IN THIS CODE BLOCK

def encode_ordinal_features(df):
    df = df.copy()
    with pd.option_context("future.no_silent_downcasting", True):
        def rep(f, d):
            df[f] = df[f].replace(d).astype('float64')
        rep('LotShape', {'Reg':0, 'IR1':1, 'IR2':2, 'IR3': 3})
        rep('LandSlope', {'Gtl':0, 'Mod':1, 'Sev':2})
        rating_dict = {'Po':0, 'Fa':1, 'TA':2, 'Gd':3, 'Ex':4}
        rep('ExterQual', rating_dict)
        rep('BsmtQual', {'NA':0, 'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5})
        rep('BsmtCond', {'NA':0, 'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5})
        rep('BsmtExposure', {'NA':0, 'No':1, 'Mn':2, 'Av':3, 'Gd':4})
        rep('BsmtFinType1', {'NA':0, 'Unf':1, 'LwQ':2, 'Rec':3, 'BLQ':4, 'ALQ':5, 'GLQ':6})
        rep('BsmtFinType2', {'NA':0, 'Unf':1, 'LwQ':2, 'Rec':3, 'BLQ':4, 'ALQ':5, 'GLQ':6})
        rep('ExterCond', rating_dict)
        rep('HeatingQC', rating_dict)
        rep('CentralAir', {'N':0, 'Y':1})
        rep('Electrical', {'Mix':0, 'FuseP':1, 'FuseF':2, 'FuseA':3, 'SBrkr':4})
        rep('KitchenQual', rating_dict)
        rep('Functional', {'Sal':0, 'Sev':1, 'Maj2':2, 'Maj1':3, 'Mod':4, 'Min2':5, 'Min1':6, 'Typ':7})
        rep('GarageFinish', {'NA':0, 'Unf':1, 'RFn':2, 'Fin':3})
        rep('GarageQual', {'NA':0, 'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5})
        rep('GarageCond', {'NA':0, 'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5})
        rep('PavedDrive', {'N':0, 'P':1, 'Y':2})
        rep('GarageType', {'CarPort':0, 'Basment':1, 'Detchd':1, 'Attchd':2, '2Types':2, 'BuiltIn':4})
        return df
        
train_df = encode_ordinal_features(train_df)
test_df = encode_ordinal_features(test_df)

# Log Transform Sale Price

In [10]:
def log_transform(df):
    df = df.copy()
    df['SalePriceTransformed'] = np.log(df['SalePrice'])
    return df

train_df = log_transform(train_df)

# Missing Data Analysis and Feature Imputation

In [11]:
# Impute missing data

def impute_missing_numerical_data(train, test):
    df = train.copy()
    test = test.copy()
    features_to_select = list(set(input_features) - set(nominal_features))
    other_features = list(set(input_features) - set(features_to_select))
    df_sel = df.filter(features_to_select)
    test_sel = test_df.filter(features_to_select)
    imp = KNNImputer()
    imp.fit(df_sel)
    imputed = imp.transform(df_sel)
    test_imputed = imp.transform(test_sel)
    df.drop(columns = features_to_select, inplace=True)
    test.drop(columns = features_to_select, inplace=True)
    df = pd.concat([df, pd.DataFrame(imputed, columns = features_to_select)], axis=1)
    test = pd.concat([test, pd.DataFrame(test_imputed, columns = features_to_select)], axis=1)
    df = df[['Id'] + input_features + output_features]
    test = test[['Id'] + input_features]
    return df, test

def impute_missing_nominal_data(train, test):
    df = train.copy()
    test = test.copy()
    df_sel = df.filter(nominal_features)
    test_sel = test.filter(nominal_features)
    imp = SimpleImputer(strategy = 'most_frequent')
    imp.fit(df_sel)
    imputed = imp.transform(df_sel)
    test_imputed = imp.transform(test_sel)
    imputed_df = pd.DataFrame(imputed, columns = df_sel.columns)
    test_imputed_df = pd.DataFrame(test_imputed, columns = test_sel.columns)
    df.drop(columns = nominal_features, inplace=True)
    test.drop(columns = nominal_features, inplace=True)
    df = pd.concat([df, imputed_df], axis=1)
    test = pd.concat([test, test_imputed_df], axis=1)
    df = df[['Id'] + input_features + output_features]
    test = test[['Id'] + input_features]
    return df, test

train_df, test_df = impute_missing_numerical_data(train_df, test_df)
train_df, test_df = impute_missing_nominal_data(train_df, test_df)

# Derived Features

In [12]:
# TODO Derived Features
derived_features = []

# Create derived features for train_df AND test_df here
input_features = ordinal_features + nominal_features + numerical_features + derived_features

# Creating Random Numbers for Use in KFold

In [13]:
NUM_OUTER_FOLDS = 5
NUM_INNER_FOLDS = 3
MAX_SEED_VAL = 2**32 - 1

def gen_kfolds():
    random.seed()
    outer_k_fold = KFold(n_splits = NUM_OUTER_FOLDS, shuffle = True, random_state = random.randrange(MAX_SEED_VAL))
    inner_k_folds = [KFold(n_splits = NUM_INNER_FOLDS, shuffle = True, random_state = random.randrange(MAX_SEED_VAL)) for _ in range(NUM_OUTER_FOLDS)]
    return outer_k_fold, inner_k_folds

OUTER_K_FOLD, INNER_K_FOLDS = gen_kfolds()


# Ordinal Encoding of Nominal Features for Random Forest and XGBoost Models

In [14]:
def ordinal_encode_nominal_features(train, test):
    df = train.copy()
    test = test.copy()
    df_nominal = df.filter(nominal_features)
    test_nominal = test.filter(nominal_features)
    if df_nominal.empty:
        return df
    ordinal_encoder = OrdinalEncoder()
    ordinal_encoder.fit(df_nominal)
    encoded = ordinal_encoder.transform(df_nominal)
    test_encoded = ordinal_encoder.transform(test_nominal)
    for i, nominal_feature in enumerate(nominal_features):
        df[nominal_feature] = encoded[:, i]
        test[nominal_feature] = test_encoded[:, i]
    return df, test

def target_encode_nominal_features(train, test):
    df = train.copy()
    test = test.copy()
    df_nominal = df.filter(nominal_features)
    test_nominal = test.filter(nominal_features)
    if df_nominal.empty:
        return df
    t_enc = TargetEncoder()
    t_enc.fit(df_nominal, df['SalePriceTransformed'])
    encoded = t_enc.transform(df_nominal)
    test_encoded = t_enc.transform(test_nominal)
    df.drop(columns = nominal_features, inplace=True)
    test.drop(columns = nominal_features, inplace=True)
    df = pd.concat([pd.DataFrame(encoded, columns = df_nominal.columns), df], axis=1)
    test = pd.concat([pd.DataFrame(test_encoded, columns = test_nominal.columns), test], axis=1)
    df = df[['Id'] + input_features + ['SalePrice', 'SalePriceTransformed']]
    test = test[['Id'] + input_features]
    return df, test

train_df_ordinal, test_df_ordinal = ordinal_encode_nominal_features(train_df, test_df)
train_df_target, test_df_target = target_encode_nominal_features(train_df, test_df)


# One Hot Encoding of Nominal Features for Neural Network Model

In [15]:
# Do One-Hot Encodings for Neural Network IN THIS CODE BLOCK
# Do operations on copy of data to avoid messing up original train_df
def one_hot_encoding(train, test):
    df = train.copy()
    test = test.copy()
    df_nominal = df.filter(nominal_features)
    test_nominal = test.filter(nominal_features)
    if df_nominal.empty:
        return df
    ohe = OneHotEncoder(sparse_output=False)
    ohe.fit(df_nominal)
    encoded = ohe.transform(df_nominal)
    test_encoded = ohe.transform(test_nominal)
    ohe_feature_names = ohe.get_feature_names_out(nominal_features)
    df.drop(columns = nominal_features, inplace = True)
    test.drop(columns = nominal_features, inplace = True)
    encoded_as_df = pd.DataFrame(encoded, columns = ohe_feature_names, index = df.index)
    test_encoded_as_df = pd.DataFrame(test_encoded, columns = ohe_feature_names, index = test.index)
    df = pd.concat([df, encoded_as_df], axis=1)
    test = pd.concat([test, test_encoded_as_df], axis=1)
    other_features = ordinal_features + numerical_features
    input_features = other_features + ohe_feature_names.tolist()
    df = df[['Id'] + input_features + ['SalePrice', 'SalePriceTransformed']]
    test = test[['Id'] + input_features]
    return df, test, input_features

train_df_ohe, test_df_ohe, ohe_input_features = one_hot_encoding(train_df, test_df)

# Outlier Analysis (Unused)

In [16]:
def iqr_outlier_removal(df):
    desc = df['SalePriceTransformed'].describe()
    q1 = desc['25%']
    q3 = desc['75%']
    iqr = q3 - q1
    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr
    outliers_removed = df.loc[(df['SalePriceTransformed'] >= lower_limit) & (df['SalePriceTransformed'] <= upper_limit)]
    return outliers_removed

In [17]:
def remove_outliers(df, *, iqr):
    iso_forest = IsolationForest()
    iso_forest.fit(train_df_ohe.filter(ohe_input_features + ['SalePriceTransformed']))
    outlier_predicts = iso_forest.predict(train_df_ohe.filter(ohe_input_features + ['SalePriceTransformed']))
    # print(outlier_predicts[outlier_predicts == -1].shape)
    good = df.loc[outlier_predicts != -1]
    if iqr:
        good = iqr_outlier_removal(good)
    return good

# fig, axes = plt.subplots(nrows=3, sharey=True, sharex=True)
# sns.histplot(better_train['SalePriceTransformed'], ax=axes[0])
# sns.histplot(good_train['SalePriceTransformed'], ax=axes[1])
# sns.histplot(train_df['SalePriceTransformed'], ax = axes[2])
# fig.tight_layout()

# Scaling data for NN

In [18]:
def scale_data(train, test, *, scaler_factory, ohe, ordinal):
    df = train.copy()
    test = test.copy()
    if ohe:
        inp_feats = ohe_input_features
    elif ordinal:
        inp_feats = numerical_features + ordinal_features
    else:
        inp_feats = input_features
    df_input = df.filter(inp_feats)
    test_input = test.filter(inp_feats)
    scaler = scaler_factory()
    scaler.fit(df_input)
    scaled = scaler.transform(df_input)
    test_scaled = scaler.transform(test_input)
    df.drop(columns = inp_feats, inplace=True)
    test.drop(columns = inp_feats, inplace=True)
    df = pd.concat([df, pd.DataFrame(scaled, index = df.index, columns = inp_feats)], axis=1)
    test = pd.concat([test, pd.DataFrame(test_scaled, index = test.index, columns = inp_feats)], axis=1)
    if ordinal:
        df = df[['Id'] + inp_feats + nominal_features + output_features]
        test = test[['Id'] + inp_feats + nominal_features]
    else:
        df = df[['Id'] + inp_feats + output_features]
        test = test[['Id'] + inp_feats]
    return df, test
    
train_df_ohe_scaled, test_df_ohe_scaled = scale_data(train_df_ohe, test_df_ohe, scaler_factory = StandardScaler, ohe=True, ordinal=False)
train_df_target_scaled, test_df_target_scaled = scale_data(train_df_target, test_df_target, scaler_factory = StandardScaler, ohe=False, ordinal=False)
train_df_ordinal_scaled, test_df_ordinal_scaled = scale_data(train_df_ordinal, test_df_ordinal, scaler_factory = StandardScaler, ohe=False, ordinal=True)


# GENERIC FETCH AND ENCODE DATA (Unused for now)

In [19]:
def get_real_train_test():
    # Import Data
    train_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
    test_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")
    return train_df, test_df

def preprocess_data(train, test):
    train, test = remove_missing_features(train, test)
    train = cyclic_encoding(train)
    test = cyclic_encoding(test)
    train = encode_ordinal_features(train)
    test = encode_ordinal_features(test)
    train = log_transform(train)
    train, test = impute_missing_numerical_data(train, test)
    train, test = impute_missing_nominal_data(train, test)
    return train, test

def encode_data(train, test):
    encoded_data = {}
    train_ordinal, test_ordinal = ordinal_encode_nominal_features(train, test)
    train_target, test_target = ordinal_encode_nominal_features(train, test)
    train_ohe, test_ohe = one_hot_encoding(train, test)
    train_ohe_scaled, test_ohe_scaled = scale_data(train_ohe, test_ohe, scaler_factory = StandardScaler, ohe=True, ordinal=False)
    train_target_scaled, test_target_scaled = scale_data(train_ohe, test_ohe, scaler_factory = StandardScaler, ohe=False, ordinal=False)
    train_ordinal_scaled, test_ordinal_scaled = scale_data(train_ordinal, test_ordinal, scaler_factory = StandardScaler, ohe=False, ordinal=True)
    encoded_data['ordinal'] = (train_ordinal, test_ordinal)
    encoded_data['target'] = (train_target, test_target)
    encoded_data['ohe'] = (train_ohe, test_ohe)
    encoded_data['ohe_scaled'] = (train_ohe_scaled, test_ohe_scaled)
    encoded_data['target_scaled'] = (train_target_scaled, test_target_scaled)
    encoded_data['ordinal_scaled'] = (train_ordinal_scaled, test_ordinal_scaled)
    return encoded_data

# Function that Trains a given model given an estimator factory, search space, and (optionally) number of iterations for bayesian optimization

In [20]:
def find_best_hyperparameters(train_x, train_y, estimator_factory, search_space, n_iter=50):
    best_hyperparams_and_scores = []
    for i, (train_indices, val_indices) in enumerate(OUTER_K_FOLD.split(X = train_x, y = train_y)):
        print(f'Outer Fold Number {i}')
        est = estimator_factory()
        #print(est, flush=True)
        train_data_x = train_x[train_indices]
        train_data_y = train_y[train_indices]
        val_data_x = train_x[val_indices]
        val_data_y = train_y[val_indices]
        bayes_search = BayesSearchCV(estimator=est, search_spaces=search_space, n_iter=n_iter, cv= INNER_K_FOLDS[i], n_jobs=-1)
        #print(bayes_search, flush=True)
        bayes_search.fit(train_data_x, train_data_y)
        best_estimator = bayes_search.best_estimator_
        best_estimator_val_score = best_estimator.score(val_data_x, val_data_y)
        
        best_hyperparams = bayes_search.best_params_
        print(f'Best hyperparams: {best_hyperparams}\nBest score: {best_estimator_val_score}\n')
        best_hyperparams_and_scores.append((best_hyperparams, best_estimator_val_score))
    best_hyperparameters = max(best_hyperparams_and_scores, key = lambda x: x[1])[0]
    return best_hyperparameters
    # best_estimator = estimator_factory()
    # best_estimator.set_params(**best_hyperparameters)
    # best_estimator.fit(train_x, train_y)
    # return best_estimator

def build_and_fit_model(train_x, train_y, estimator_factory, hp, **fit_kwargs):
    best_estimator = estimator_factory()
    best_estimator.set_params(**hp)
    best_estimator.fit(train_x, train_y, **fit_kwargs)
    return best_estimator

In [21]:
def get_x_for_encoding(df, *, encoding):
    if encoding == "target":
        x = df.filter(input_features).to_numpy()
    elif encoding == "ohe":
        x = df.filter(ohe_input_features).to_numpy()
    else:
        raise Exception(f'Unexpected encoding: {encoding}')
    return x

# INSERT IMPLEMENTATION OF RANDOM FOREST HERE

**We should have a sklearn estimator as well as a well-defined search space in scope at the end of this section**

In [22]:
rf_search_space = {
    'n_estimators': Integer(50, 300), #Integer(50, 100),
    'max_depth': Integer(3, 20), #Integer(3, 10),
    'min_samples_leaf': Integer(1, 10)
}

def find_best_hyperparameters_for_rf(train, *, encoding):
    df = train
    train_x = get_x_for_encoding(train, encoding=encoding)
    train_y = df['SalePriceTransformed'].to_numpy()
    best_hp = find_best_hyperparameters(train_x, train_y, RandomForestRegressor, rf_search_space, n_iter=50)
    return best_hp
    

def train_rf(train, best_hp, *, encoding):
    df = train
    train_x = get_x_for_encoding(train, encoding=encoding)
    train_y = df['SalePriceTransformed'].to_numpy()
    # best_hp = find_best_hyperparameters(train_x, train_y, RandomForestRegressor, search_space, n_iter=50)
    est_factory = lambda: RandomForestRegressor(oob_score=True)
    rf = build_and_fit_model(train_x, train_y, est_factory, best_hp)
    return rf

def find_best_hp_for_and_train_rf(train, *, encoding):
    best_hp = find_best_hyperparameters_for_rf(train, encoding=encoding)
    rf = train_rf(train, best_hp, encoding=encoding)
    return rf, best_hp

def evaluate_rf(train, rf, *, encoding):
    oob_sc = rf.oob_score_
    mse = mean_squared_error(y_true = train['SalePriceTransformed'], y_pred = rf.predict(get_x_for_encoding(train, encoding=encoding)))
    sc = rf.score(get_x_for_encoding(train, encoding=encoding), train['SalePriceTransformed'])
    return dict(oob_score=oob_sc, mse=mse, score=sc)
    
#rf = train_rf(train_df_ordinal)
#rf = train_rf(train_df_target)

In [23]:
#target_rf, best_hp_for_rf_target = find_best_hp_for_and_train_rf(train_df_target, encoding='target')
# ohe_rf, best_hp_for_rf_ohe = find_best_hp_for_and_train_rf(train_df_ohe, encoding='ohe')
# TODO TODO TODO We may just want to save the best hyperparameters for rf in a cell (manually type out the dict) so that we don't have to the the k-fold cross validation and Bayesian optimization every time

In [24]:
BEST_RF_TARGET_HYPERPARAMETERS = {
      'max_depth':18
    , 'min_samples_leaf':1
    , 'n_estimators':299
}
BEST_RF_OHE_HYPERPARAMETERS = {
      'max_depth':19 # TODO FIX
    , 'min_samples_leaf':1 # TODO FIX
    , 'n_estimators':299 # TODO FIX
}

In [25]:
rf_target = train_rf(train_df_target, BEST_RF_TARGET_HYPERPARAMETERS, encoding='target')
rf_ohe = train_rf(train_df_ohe, BEST_RF_OHE_HYPERPARAMETERS, encoding='ohe')
rf_target_train_results = evaluate_rf(train_df_target, rf_target, encoding='target')
rf_ohe_train_results = evaluate_rf(train_df_ohe, rf_ohe, encoding='ohe')
with open("./rf_train_results.txt", "w") as f:
    f.write(f'rf_target_train_results: {rf_target_train_results}\n')
    f.write(f'rf_ohe_train_results: {rf_ohe_train_results}\n')
rf_target_preds = rf_target.predict(get_x_for_encoding(test_df_target, encoding='target'))
rf_ohe_preds = rf_ohe.predict(get_x_for_encoding(test_df_ohe, encoding='ohe'))
write_preds_to_file("./rf_preds_target.csv", np.exp(rf_target_preds), test_df_target, is_nn=False)
write_preds_to_file("./rf_preds_ohe.csv", np.exp(rf_ohe_preds), test_df_ohe, is_nn=False)

In [26]:
# with open("./rf_train_results.txt", "w") as f:
#     f.write(f'rf_target_train_results: {rf_target_train_results}\n')
#     f.write(f'rf_ohe_train_results: {rf_ohe_train_results}\n')

In [27]:
# rf_target_preds = rf_target.predict(get_x_for_encoding(test_df_target, encoding='target'))
# rf_ohe_preds = rf_ohe.predict(get_x_for_encoding(test_df_ohe, encoding='ohe'))

In [28]:
# write_preds_to_file("./rf_preds_target.csv", np.exp(rf_target_preds), test_df_target, is_nn=False)
# write_preds_to_file("./rf_preds_ohe.csv", np.exp(rf_ohe_preds), test_df_ohe, is_nn=False)

In [29]:
#best_hp_for_rf = find_best_hyperparameters_for_rf(train_df_target)
# def try_rand_forest():
#     tr, _ = get_real_train_test()
#     split = train_test_split(tr)
#     tr = split[0]
#     te = split[1]
#     tr, te = preprocess_data(tr, te)
#     enc = encode_data(tr, te)
#     rf = train_rf(enc['target'])
#     return rf

# rf = try_rand_forest()


In [30]:
# with open("./best_random_forest_hyperparameters.txt", 'w') as f:
#     f.write(f'{rf.get_params()}')

In [31]:
# Need to modify this code
# rf = target_rf
# train_x = train_df_target.filter(input_features)
# train_y = train_df_target['SalePriceTransformed']
# std = np.std([tree.feature_importances_ for tree in rf.estimators_], axis=0)
# forest_importances = pd.Series(rf.feature_importances_, index=train_x.columns)
# forest_importances = forest_importances.sort_values()[-15:]
# fig, ax = plt.subplots(figsize = (15, 6))
# sns.barplot(x = forest_importances.index, y = forest_importances)
# print(forest_importances[-15:])
# print(forest_importances[-15:].index)
# fig.tight_layout()

In [32]:
# rf = target_rf
# train_x = train_df_target.filter(input_features)
# train_y = train_df_target['SalePriceTransformed']
# predicts = rf.predict(train_x)
# mse = mean_squared_error(y_true = train_y, y_pred = predicts)
# print(mse)
# print(rf.score(train_x, train_y))
# print(rf.oob_score_)

In [33]:
# rf = ohe_rf
# train_x = train_df_ohe.filter(ohe_input_features).to_numpy()
# train_y = train_df_ohe['SalePriceTransformed'].to_numpy()
# predicts = rf.predict(train_x)
# mse = mean_squared_error(y_true = train_y, y_pred = predicts)
# print(mse)
# print(rf.score(train_x, train_y))
# print(rf.oob_score_)

In [34]:
# rf = ohe_rf
# train_x = train_df_ohe.filter(ohe_input_features)
# train_y = train_df_ohe['SalePriceTransformed']
# std = np.std([tree.feature_importances_ for tree in rf.estimators_], axis=0)
# forest_importances = pd.Series(rf.feature_importances_, index=train_x.columns)
# forest_importances = forest_importances.sort_values()[-15:]
# fig, ax = plt.subplots(figsize = (15, 6))
# sns.barplot(x = forest_importances.index, y = forest_importances)
# print(forest_importances[-15:])
# print(forest_importances[-15:].index)
# fig.tight_layout()

# INSERT IMPLEMENTATION OF XGBOOST HERE

In [35]:
# define search space
xgb_search_space = {
    #'n_estimators': Integer(50, 100),
    'max_depth': Integer(1, 15), #Integer(3, 10),
    'gamma': (1e-8, 1e+1, 'log-uniform'),
    'learning_rate': (0.01, 0.4),
    'reg_lambda': (0.5, 2.0),
    'min_child_weight': Integer(1, 7),
    'subsample': (0.55, 1.0),
    'colsample_bytree': (0.2, 0.9)
}

def find_best_hyperparameters_for_xgboost(train, *, encoding):
    df = train
    train_x = get_x_for_encoding(train, encoding=encoding)
    train_y = df['SalePriceTransformed'].to_numpy()
    best_hp = find_best_hyperparameters(train_x, train_y, XGBRegressor, xgb_search_space, n_iter=50)
    return best_hp

def train_xgboost(train, best_hp, *, encoding):
    df = train
    train_x = get_x_for_encoding(train, encoding=encoding)
    train_y = df['SalePriceTransformed']
    xgb = build_and_fit_model(train_x, train_y, XGBRegressor, best_hp)
    return xgb

def find_best_hp_for_and_train_xgboost(train, *, encoding):
    best_hp = find_best_hyperparameters_for_xgboost(train, encoding=encoding)
    xgb = train_xgboost(train, best_hp, encoding=encoding)
    return xgb, best_hp

def evaluate_xgb(train, xgb, *, encoding):
    mse = mean_squared_error(y_true = train['SalePriceTransformed'], y_pred = xgb.predict(get_x_for_encoding(train, encoding=encoding)))
    sc = xgb.score(get_x_for_encoding(train, encoding=encoding), train['SalePriceTransformed'])
    return dict(mse=mse, score=sc)

# output the predictions in the Kaggle competition format
# xgb_predictions = xgb.predict(test_x)
# f = open("xgb_preds.csv", "a")
# f.write("Id,SalePrice")
# for i in range(len(test_data.index)):
#     f.write(test_data.iloc[i].get("Id") + "," + xgb_predictions[i])
# f.close()

In [36]:
# xgb_target, best_hp_for_xgboost_target = find_best_hp_for_and_train_xgboost(train_df_target, encoding='target')
# xgb_ohe, best_hp_for_xgboost_ohe = find_best_hp_for_and_train_xgboost(train_df_ohe, encoding='ohe')

In [37]:
BEST_XGB_TARGET_HYPERPARAMETERS = {
    'colsample_bytree': 0.8674926647227033,
    'gamma': 0.0001984173974768878,
    'learning_rate': 0.1,
    'max_depth': 15,
    'min_child_weight':10,
    'reg_lambda':1.4416378817611377,
    'subsample':0.5
}

# BEST_XGB_TARGET_HYPERPARAMETERS2 = {
#     'colsample_bytree': 0.39778202671885776,
#     'gamma': 0.002253514315613315,
#     'learning_rate': 0.06938083392348873,
#     'max_depth': 15,
#     'min_child_weight':6,
#     'reg_lambda':2.0,
#     'subsample':0.55
# }

BEST_XGB_OHE_HYPERPARAMETERS = {
    'colsample_bytree': 0.3831226978013727,
    'gamma': 0.00043546709402603136,
    'learning_rate': 0.23370000596285093,
    'max_depth': 2,
    'min_child_weight':2,
    'reg_lambda':1.690785330143059,
    'subsample':0.9834195351879633
}

In [38]:
xgb_target = train_xgboost(train_df_target, BEST_XGB_TARGET_HYPERPARAMETERS, encoding='target')
xgb_ohe = train_xgboost(train_df_ohe, BEST_XGB_OHE_HYPERPARAMETERS, encoding='ohe')

In [39]:
xgb_target_train_results = evaluate_xgb(train_df_target, xgb_target, encoding='target')
xgb_ohe_train_results = evaluate_xgb(train_df_ohe, xgb_ohe, encoding='ohe')

In [40]:
with open("./xgb_train_results.txt", "w") as f:
    f.write(f'xgb_target_train_results: {xgb_target_train_results}\n')
    f.write(f'xgb_ohe_train_results: {xgb_ohe_train_results}\n')

In [41]:
xgb_target = train_xgboost(train_df_target, BEST_XGB_TARGET_HYPERPARAMETERS, encoding='target')
xgb_ohe = train_xgboost(train_df_ohe, BEST_XGB_OHE_HYPERPARAMETERS, encoding='ohe')
xgb_target_train_results = evaluate_xgb(train_df_target, xgb_target, encoding='target')
xgb_ohe_train_results = evaluate_xgb(train_df_ohe, xgb_ohe, encoding='ohe')
with open("./xgb_train_results.txt", "w") as f:
    f.write(f'xgb_target_train_results: {xgb_target_train_results}\n')
    f.write(f'xgb_ohe_train_results: {xgb_ohe_train_results}\n')
xgb_target_preds = xgb_target.predict(get_x_for_encoding(test_df_target, encoding='target'))
xgb_ohe_preds = xgb_ohe.predict(get_x_for_encoding(test_df_ohe, encoding='ohe'))
write_preds_to_file("./xgb_preds_target.csv", np.exp(xgb_target_preds), test_df_target, is_nn=False)
write_preds_to_file("./xgb_preds_ohe.csv", np.exp(xgb_ohe_preds), test_df_ohe, is_nn=False)

In [42]:
# best_hp_for_xgboost_ohe
# BEST_XGB_OHE_HYPERPARAMETERS = {
#     'colsample_bytree': 0.22969457938770133,
#     'gamma': 0.010316164334201692,
#     'learning_rate': 0.1,
#     'max_depth': 15,
#     'min_child_weight':10,
#     'reg_lambda':0.5,
#     'subsample':0.8731603648422741
# }

In [43]:
# xgb_ohe = train_xgboost(train_df_ohe, best_hp = BEST_XGB_OHE_HYPERPARAMETERS, encoding='ohe')

In [44]:
# # xgb = xgb_target # CHANGE TO xgb_ohe to look at ohe case
# train_x = train_df_target.filter(input_features)
# train_y = train_df_target['SalePriceTransformed']
# predicts = xgb.predict(train_x)
# mse = mean_squared_error(y_true = train_y, y_pred = predicts)
# print(mse)
# xgb_score = xgb.score(train_x, train_y)
# print(xgb_score)

In [45]:
# xgb = xgb_ohe
# train_x = train_df_ohe.filter(ohe_input_features)
# train_y = train_df_ohe['SalePriceTransformed']
# predicts = xgb.predict(train_x)
# mse = mean_squared_error(y_true = train_y, y_pred = predicts)
# print(mse)
# xgb_score = xgb.score(train_x, train_y)
# print(xgb_score)

In [46]:
# with open("./best_xgboost_hyperparameters2.txt", 'w') as f:
#     f.write(f'{xgb.get_params()} with score {xgb_score}')

# INSERT IMPLEMENTATION OF NEURAL NETWORK HERE

For data normalization we use keras' Normalization layer.

**We should have a sklearn estimator as well as a well-defined search space in scope at the end of this section**

# Neural Network Without Embeddings

In [47]:
class MyHyperModelSimple(keras_tuner.HyperModel):
    def __init__(self, *, encoding):
        super().__init__()
        if encoding not in ['target', 'ohe']:
            raise(Exception(f'Unexpected encoding type: {encoding}'))
        self.encoding = encoding

    def build(self, hp):   
        dropout_rate = hp.Float("dropout_rate", min_value = 0.15, max_value = 0.5)
        learning_rate = hp.Float("learning_rate", min_value = 0.0005, max_value = 0.1)
        batch_norm_momentum = hp.Float("batch_norm_momentum", min_value = 0.985, max_value = 0.995)
        num_hid_layers = hp.Int("num_hidden_layers", min_value = 2, max_value = 10)
        hid_node_counts = []
        for i in range(10):
            hid_nodes = hp.Int(f'hid_nodes{i}', min_value = 64, max_value = 1024, step = 2, sampling = 'log', parent_name = "num_hidden_layers", parent_values = list(range(max(i + 1, 2), 10 + 1)))
            if i < num_hid_layers:
                hid_node_counts.append(hid_nodes)
        num_input_features = len(input_features) if self.encoding == 'target' else len(ohe_input_features)
        input = keras.layers.Input((num_input_features,), name = 'inputs')
        output = input
        for i, num_nodes in enumerate(hid_node_counts):
            output = keras.layers.Dense(num_nodes, activation = 'relu', name=f'dense_{i}')(output)
            output = keras.layers.BatchNormalization(momentum = batch_norm_momentum, name=f'batch_norm_{i}')(output)
            output = keras.layers.Dropout(dropout_rate, name = f'dropout_{i}')(output)
        output = keras.layers.Dense(1, activation = 'linear', name=f'output')(output)
        adam = keras.optimizers.Adam(learning_rate = learning_rate)
        model = keras.Model(input, output)
        model.compile(optimizer=adam, loss='mse', metrics = [keras.metrics.MeanSquaredError(), keras.metrics.R2Score()])
        return model
    def fit(self, hp, model, *args, **kwargs):
        return model.fit(*args, shuffle=True, **kwargs)



def find_best_hyperparameters_for_nn_simple(train, *, epochs=100, encoding):
    def mk_tuner_for_simple():
        obj = keras_tuner.Objective('val_mean_squared_error', direction = 'min')
        hypermodel = MyHyperModelSimple(encoding=encoding)
        tuner = keras_tuner.BayesianOptimization(hypermodel = hypermodel, objective = obj, max_trials = 10, overwrite=True)
        return tuner
    best_hyperparams_and_scores = []
    train_x = get_x_for_encoding(train, encoding=encoding)
    train_y = train['SalePriceTransformed'].to_numpy()
    for i, (train_indices, val_indices) in enumerate(OUTER_K_FOLD.split(X = train_x, y = train_y)):
        #print(f'Outer Fold Number {i}')
        train_data_x = train_x[train_indices]
        train_data_y = train_y[train_indices]
        val_data_x = train_x[val_indices]
        val_data_y = train_y[val_indices]
        nested_best_hyperparams_and_scores_and_models = []
        tuner = mk_tuner_for_simple()
        for j, (train_indices2, val_indices2) in enumerate(INNER_K_FOLDS[i].split(X = train_data_x, y = train_data_y)):
            #print(f'Outer Fold Number {i}, Inner Fold Number {j}')
            tx = train_data_x[train_indices2]
            ty = train_data_y[train_indices2]
            vx = train_data_x[val_indices2]
            vy = train_data_y[val_indices2]
            tuner.search(tx, ty, epochs=epochs, validation_data=(vx, vy))
            best_hp = tuner.get_best_hyperparameters()[0]
            best_model = tuner.hypermodel.build(best_hp)
            hypermodel = MyHyperModelSimple(encoding=encoding)
            hypermodel.fit(best_hp, best_model, train_data_x, train_data_y, epochs=epochs)
            mse_val = best_model.evaluate(val_data_x, val_data_y)[0]
            nested_best_hyperparams_and_scores_and_models.append((best_hp, mse_val, best_model))
            # with open(f'./temp/{encoding}/outer_{i}_inner_{j}.txt', 'w') as f:
            #     f.write(f'{(best_hp, mse_val)}')
            
        (best_hp, best_mse_val, _) = min(nested_best_hyperparams_and_scores_and_models, key = lambda x: x[1])
        #print(f'Best hyperparams: {best_hp}\n Best score: {best_mse_val}\n')
        best_hyperparams_and_scores.append((best_hp, best_mse_val))
    #best_hp = min(best_hyperparams_and_scores, key = lambda x: x[1])[0]
    return best_hyperparams_and_scores

def train_nn_simple(train, best_hp, *, epochs=100, encoding, **fit_kwargs):
    hp_model = MyHyperModelSimple(encoding=encoding)
    model = hp_model.build(best_hp)
    train_x = get_x_for_encoding(train, encoding=encoding)
    train_y = train['SalePriceTransformed']
    model.fit(train_x, train_y, epochs=epochs, **fit_kwargs)
    return model

def get_best_hyperparameter(best_hps_and_scores):
    best_hp = min(best_hps_and_scores, key = lambda x: x[1])[0]
    return best_hp


def find_best_hp_for_and_train_nn_simple(train, *, encoding):
    best_hp = get_best_hyperparameter(find_best_hyperparameters_for_nn_simple(train, encoding=encoding))
    nn_simple = train_nn_simple(train, best_hp, encoding=encoding)
    return nn_simple, best_hp

def evaluate_simple_nn(*, model, data, encoding):
    x = get_x_for_encoding(data, encoding=encoding)
    y = data['SalePriceTransformed']
    res = model.evaluate(x, y)
    return res
# best_nn_no_embeddings_target = train_nn_simple(train_df_target_scaled, direc="./simple_keras_tuner", encoding='target')
#best_nn_no_embeddings_ohe = train_nn_simple(train_df_ohe_scaled, direc="./simple_keras_tuner_ohe", encoding='ohe')


In [48]:
# best_nn_simple_target, best_hp_for_simple_nn_target = find_best_hp_for_and_train_nn_simple(train_df_target_scaled, encoding='target')
# nn_simple_target = train_nn_simple(train_df_target_scaled, best_hp_for_simple_nn_target, encoding='target')

# for (hp, score) in best_hps_and_scores_ohe:
#     print(f'Score: {score}\nValues:{hp.values}')

# best_hp = get_best_hyperparameter(best_hp_and_scores_embedding)

# evaluate_simple_nn(model=nn_simple_ohe, data=train_df_ohe_scaled, encoding='ohe')

# nn_simple_ohe = train_nn_simple(train_df_ohe_scaled, best_hp, encoding='ohe', epochs=100)

# # best_nn_simple_ohe, best_hp_for_simple_nn_ohe = find_best_hp_for_and_train_nn_simple(train_df_ohe_scaled, encoding='ohe')
# best_hps_and_scores_ohe = find_best_hyperparameters_for_nn_simple(train_df_ohe_scaled, encoding='ohe')

# evaluate_simple_nn(model=nn_simple_target, data=train_df_target_scaled, encoding='target')

# best_hp_for_simple_nn_target.values

In [49]:
def get_precomputed_best_hp_for_simple_nn_target():
    hp = keras_tuner.HyperParameters()
    hp.Float("dropout_rate", min_value = 0.15, max_value = 0.5)
    hp.Float("learning_rate", min_value = 0.0005, max_value = 0.1)
    hp.Float("batch_norm_momentum", min_value = 0.985, max_value = 0.995)
    hp.Int("num_hidden_layers", min_value = 2, max_value = 10)
    hp.Int("hid_nodes0", min_value = 2, max_value = 10)
    hp.Int("hid_nodes1", min_value = 2, max_value = 10)
    hp.Int("hid_nodes2", min_value = 2, max_value = 10)
    hp.Int("hid_nodes3", min_value = 2, max_value = 10)
    hp.Int("hid_nodes4", min_value = 2, max_value = 10)
    hp.Int("hid_nodes5", min_value = 2, max_value = 10)
    hp.values['dropout_rate'] = 0.476979453991086
    hp.values['learning_rate'] = 0.08259772224638878
    hp.values['batch_norm_momentum'] = 0.9903974622372896
    hp.values['num_hid_layers'] = 6
    hid_node_cts = [128, 256, 128, 512, 512, 64]
    for i in range(6):
        hp.values[f'hid_nodes{i}'] = hid_node_cts[i]
    return hp

def get_precomputed_best_hp_for_simple_nn_ohe():
    hp = keras_tuner.HyperParameters()
    hp.Float("dropout_rate", min_value = 0.15, max_value = 0.5)
    hp.Float("learning_rate", min_value = 0.0005, max_value = 0.1)
    hp.Float("batch_norm_momentum", min_value = 0.985, max_value = 0.995)
    hp.Int("num_hidden_layers", min_value = 2, max_value = 10)
    hp.Int("hid_nodes0", min_value = 2, max_value = 10)
    hp.Int("hid_nodes1", min_value = 2, max_value = 10)
    hp.values['dropout_rate'] = 0.2809579573235204
    hp.values['learning_rate'] = 0.08383867518545239
    hp.values['batch_norm_momentum'] = 0.9939429375240272
    hp.values['num_hid_layers'] = 2
    hid_node_cts = [128, 64]
    for i in range(2):
        hp.values[f'hid_nodes{i}'] = hid_node_cts[i]
    return hp


    
    
BEST_HP_FOR_SIMPLE_NN_TARGET = get_precomputed_best_hp_for_simple_nn_target()
BEST_HP_FOR_SIMPLE_NN_OHE = get_precomputed_best_hp_for_simple_nn_ohe()

In [50]:
best_nn_simple_target = train_nn_simple(train_df_target_scaled, BEST_HP_FOR_SIMPLE_NN_TARGET, encoding='target')
best_nn_simple_ohe = train_nn_simple(train_df_ohe_scaled, BEST_HP_FOR_SIMPLE_NN_OHE, encoding='ohe')

Epoch 1/100


2025-04-24 01:39:04.498649: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


46/46 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 51.3173 - mean_squared_error: 51.3173 - r2_score: -312.6418
Epoch 2/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.3462 - mean_squared_error: 2.3462 - r2_score: -13.9470
Epoch 3/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.0799 - mean_squared_error: 1.0799 - r2_score: -5.6154
Epoch 4/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.9597 - mean_squared_error: 0.9597 - r2_score: -5.4729
Epoch 5/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.8974 - mean_squared_error: 0.8974 - r2_score: -4.6414
Epoch 6/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.9472 - mean_squared_error: 0.9472 - r2_score: -4.7490
Epoch 7/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.7290 - mean_squared_error: 0.7290 - r2_score: -3.7208
Epoch 8/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.6635 - mean_squared_error: 0.6635 - r2_score: -3.0877
Epoch 9/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.1323 - mean_squared_error:

In [51]:
nn_simple_target_results = evaluate_simple_nn(model=best_nn_simple_target, data=train_df_target_scaled, encoding='target')
nn_simple_ohe_results = evaluate_simple_nn(model=best_nn_simple_ohe, data=train_df_ohe_scaled, encoding='ohe')

46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0189 - mean_squared_error: 0.0189 - r2_score: 0.8846
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0067 - mean_squared_error: 0.0067 - r2_score: 0.9591


In [52]:
with open("./simple_nn_train_results.txt", "w") as f:
    f.write(f'simple_nn_target_train_results: {nn_simple_target_results}\n')
    f.write(f'simple_nn_ohe_train_results: {nn_simple_ohe_results}\n')

In [53]:
simple_nn_target_preds = best_nn_simple_target.predict(get_x_for_encoding(test_df_target_scaled, encoding='target'))
simple_nn_ohe_preds = best_nn_simple_ohe.predict(get_x_for_encoding(test_df_ohe_scaled, encoding='ohe'))
write_preds_to_file("./simple_nn_preds_target.csv", np.exp(simple_nn_target_preds), test_df_target_scaled, is_nn=True)
write_preds_to_file("./simple_nn_preds_ohe.csv", np.exp(simple_nn_ohe_preds), test_df_ohe_scaled, is_nn=True)

46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [54]:
# best_nn_simple_ohe, best_hp_for_simple_nn_ohe = find_best_hp_for_and_train_nn_simple(train_df_ohe_scaled, encoding='ohe')
# TODO TODO TODO We may just want to save the best hyperparameters for each encoding of the simple nn in a cell (manually type out the dict) so that we don't have to the the k-fold cross validation and Bayesian optimization every time

In [55]:
# with open("./best_simple_nn_hyperparameters.txt2", 'w') as f:
#     f.write(f'{best_hyperparams_and_scores_for_simple_nn[3][0].values}')

In [56]:
#best_nn_simple_target.evaluate(train_df_target_scaled.filter(input_features).to_numpy(), train_df_target_scaled['SalePriceTransformed'].to_numpy())
# evaluate_simple_nn(model=best_nn_simple_target, data=train_df_target_scaled, encoding='target')

In [57]:
# best_nn_simple_ohe.evaluate(train_df_ohe_scaled.filter(ohe_input_features).to_numpy(), train_df_ohe_scaled['SalePriceTransformed'].to_numpy())
# evaluate_simple_nn(model=best_nn_simple_ohe, data=train_df_ohe_scaled, encoding='target')

In [58]:
# keras.utils.plot_model(best_nn_no_embeddings_target)

In [59]:
# keras.utils.plot_model(best_nn_no_embeddings_ohe)

# Neural Network With Embeddings

In [60]:
def get_precomputed_best_hp_for_nn_embedding():
    hp = keras_tuner.HyperParameters()
    hp.Float("dropout_rate", min_value = 0.15, max_value = 0.5)
    hp.Float("learning_rate", min_value = 0.0005, max_value = 0.1)
    hp.Float("batch_norm_momentum", min_value = 0.985, max_value = 0.995)
    hp.Int("num_hidden_layers", min_value = 2, max_value = 10)
    hp.Int("hid_nodes0", min_value = 2, max_value = 10)
    hp.Int("hid_nodes1", min_value = 2, max_value = 10)
    hp.Int("hid_nodes2", min_value = 2, max_value = 10)
    hp.Int("hid_nodes3", min_value = 2, max_value = 10)
    hp.Int("hid_nodes4", min_value = 2, max_value = 10)
    hp.Int("hid_nodes5", min_value = 2, max_value = 10)
    hp.values['dropout_rate'] = 0.3905778977909982
    hp.values['learning_rate'] = 0.055171199667477785
    hp.values['batch_norm_momentum'] = 0.9924157700801539
    hp.values['num_hid_layers'] = 5
    hid_node_cts = [128, 64, 256, 512, 64]
    for i in range(5):
        hp.values[f'hid_nodes{i}'] = hid_node_cts[i]
    return hp


BEST_HP_FOR_NN_EMBEDDING = get_precomputed_best_hp_for_nn_embedding()

In [61]:
all_numerical_features = numerical_features + ordinal_features
num_categories_for_nominals = (train_df_ordinal_scaled.filter(nominal_features).max() + 1).astype('int32')

class MyHyperModel(keras_tuner.HyperModel):
    def build(self, hp):
        nominal_input_list = []
        nominal_embedding_list = []
    
        dropout_rate = hp.Float("dropout_rate", min_value = 0.15, max_value = 0.5)
        learning_rate = hp.Float("learning_rate", min_value = 0.0005, max_value = 0.1)
        batch_norm_momentum = hp.Float("batch_norm_momentum", min_value = 0.985, max_value = 0.995)
        num_hid_layers = hp.Int("num_hidden_layers", min_value = 2, max_value = 10)
        hid_node_counts = []
        for i in range(10):
            hid_nodes = hp.Int(f'hid_nodes{i}', min_value = 64, max_value = 1024, step = 2, sampling = 'log', parent_name = "num_hidden_layers", parent_values = list(range(max(i + 1, 2), 10 + 1)))
            if i < num_hid_layers:
                hid_node_counts.append(hid_nodes)
        numerical_input = keras.layers.Input((len(all_numerical_features),), name = 'numerical_inputs')
        for i, nominal_feature in enumerate(nominal_features):
            nominal_input = keras.layers.Input((1,), name = f'{nominal_feature}_input')
            nominal_embedding = keras.layers.Embedding(num_categories_for_nominals.iloc[i], math.ceil(num_categories_for_nominals.iloc[i] / 2.0), name=f'{nominal_feature}_embedding')(nominal_input)
            nominal_flattened = keras.layers.Flatten(name = f'{nominal_feature}_flatten')(nominal_embedding)
            nominal_embedding_list.append(nominal_flattened)
            nominal_input_list.append(nominal_input)
        concatenated_features = keras.layers.Concatenate(name='concatenate')([numerical_input] + nominal_embedding_list)
        output = concatenated_features
        for i, num_nodes in enumerate(hid_node_counts):
            output = keras.layers.Dense(num_nodes, activation = 'relu', name=f'dense_{i}')(output)
            output = keras.layers.BatchNormalization(momentum = batch_norm_momentum, name=f'batch_norm_{i}')(output)
            output = keras.layers.Dropout(dropout_rate, name = f'dropout_{i}')(output)
        output = keras.layers.Dense(1, activation = 'linear', name=f'output')(output)
        adam = keras.optimizers.Adam(learning_rate = learning_rate)
        model = keras.Model(inputs = [numerical_input] + nominal_input_list, outputs=output)
        model.compile(optimizer=adam, loss='mse', metrics = [keras.metrics.MeanSquaredError(), keras.metrics.R2Score()])
        return model
    def fit(self, hp, model, *args, **kwargs):
        return model.fit(*args, shuffle=True, **kwargs)
        

# build_model(keras_tuner.HyperParameters())

def get_inp(x):
    inp = {}
    inp['numerical_inputs'] = x[:, :len(all_numerical_features)]
    for k, feat in enumerate(nominal_features):
        inp[f'{feat}_input'] = x[:, k + len(all_numerical_features)]
    return inp

def mk_tuner():
    obj = keras_tuner.Objective('val_mean_squared_error', direction = 'min')
    hypermodel = MyHyperModel()
    tuner = keras_tuner.BayesianOptimization(hypermodel = hypermodel, objective = obj, max_trials = 10, overwrite=True, directory="./nn_embedding_model")
    return tuner

def find_best_hyperparameters_for_nn(train, epochs=100):
    best_hyperparams_and_scores = []
    train_x = train.filter(all_numerical_features + nominal_features).to_numpy()
    train_y = train['SalePriceTransformed'].to_numpy()
    for i, (train_indices, val_indices) in enumerate(OUTER_K_FOLD.split(X = train_x, y = train_y)):
        #print(f'Outer Fold Number {i}')
        train_data_x = train_x[train_indices]
        train_data_y = train_y[train_indices]
        val_data_x = train_x[val_indices]
        val_data_y = train_y[val_indices]
        nested_best_hyperparams_and_scores_and_models = []
        tuner = mk_tuner()
        for j, (train_indices2, val_indices2) in enumerate(INNER_K_FOLDS[i].split(X = train_data_x, y = train_data_y)):
            #print(f'Outer Fold Number {i}, Inner Fold Number {j}')
            tx = train_data_x[train_indices2]
            ty = train_data_y[train_indices2]
            vx = train_data_x[val_indices2]
            vy = train_data_y[val_indices2]
            tuner.search(get_inp(tx), ty, epochs=epochs, validation_data=(get_inp(vx), vy))
            best_hp = tuner.get_best_hyperparameters()[0]
            best_model = tuner.hypermodel.build(best_hp)
            hypermodel = MyHyperModel()
            hypermodel.fit(best_hp, best_model, get_inp(train_data_x), train_data_y, epochs=epochs)
            mse_val = best_model.evaluate(get_inp(val_data_x), val_data_y)[0]
            nested_best_hyperparams_and_scores_and_models.append((best_hp, mse_val, best_model))
            # with open(f'./temp/embedding/outer_{i}_inner_{j}.txt', 'w') as f:
            #     f.write(f'{(best_hp, mse_val)}')
            
        (best_hp, best_mse_val, _) = min(nested_best_hyperparams_and_scores_and_models, key = lambda x: x[1])
        #print(f'Best hyperparams: {best_hp}\n Best score: {best_mse_val}\n')
        best_hyperparams_and_scores.append((best_hp, best_mse_val))
    #best_hp = min(best_hyperparams_and_scores, key = lambda x: x[1])[0]
    return best_hyperparams_and_scores
    #return best_hp

def train_nn(train, best_hp, epochs=100, *, shuffle):
    hp_model = MyHyperModel()
    model = hp_model.build(best_hp)
    x = train.filter(all_numerical_features + nominal_features).to_numpy()
    y = train['SalePriceTransformed'].to_numpy()
    model.fit(get_inp(x), y, epochs=epochs, shuffle=shuffle)
    return model

def evaluate_nn(*, model, data):
    x = get_inp(data.filter(all_numerical_features + nominal_features).to_numpy())
    y = data['SalePriceTransformed'].to_numpy()
    res = model.evaluate(x, y)
    return res



# nn_model, best_hp = train_nn(train_df_ordinal_scaled)


In [62]:
best_nn_embedding = train_nn(train_df_ordinal_scaled, BEST_HP_FOR_NN_EMBEDDING, shuffle=False)


Epoch 1/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 56.2073 - mean_squared_error: 56.2073 - r2_score: -345.4896
Epoch 2/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.4320 - mean_squared_error: 2.4320 - r2_score: -13.9772
Epoch 3/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.5144 - mean_squared_error: 1.5144 - r2_score: -8.3183
Epoch 4/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.5475 - mean_squared_error: 1.5475 - r2_score: -8.5078
Epoch 5/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.3560 - mean_squared_error: 1.3560 - r2_score: -7.3198
Epoch 6/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.4070 - mean_squared_error: 1.4070 - r2_score: -7.6534
Epoch 7/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.2665 - mean_squared_error: 1.2665 - r2_score: -6.7866
Epoch 8/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.3203 - mean_squared_error: 1.3203 - r2_score: -7.1201
Epoch 9/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.2801 - mean_s

In [63]:
nn_embedding_results = evaluate_nn(model=best_nn_embedding, data=train_df_ordinal_scaled)

46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0079 - mean_squared_error: 0.0079 - r2_score: 0.9513


In [64]:
with open("./nn_embedding_train_results.txt", "w") as f:
    f.write(f'nn_embedding_train_results: {nn_embedding_results}\n')

In [65]:
x = test_df_ordinal_scaled.filter(all_numerical_features + nominal_features).to_numpy()
nn_embedding_preds = best_nn_embedding.predict(get_inp(x))
write_preds_to_file("./nn_embedding_preds.csv", np.exp(nn_embedding_preds), test_df_ordinal_scaled, is_nn=True)

46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step


In [66]:
# hp_model = MyHyperModel()
# model = hp_model.build(best_hp)
# x = train_df_ordinal_scaled.filter(all_numerical_features + nominal_features).to_numpy()
# y = train_df_ordinal_scaled['SalePriceTransformed'].to_numpy()
# nn_model.fit(get_inp(x), y, epochs=100, shuffle=False)




# output the predictions in the Kaggle competition format
def write_preds_to_file(file, preds, test):
    with open(file, "w") as f:
        f.write(f"Id,SalePrice\n")
        for i in range(len(test.index)):
            f.write(f'{test.iloc[i].get("Id")},{nn_predicts_real[i][0]}\n')



In [67]:
# nn_model.evaluate(get_inp(train_df_ordinal_scaled.filter(all_numerical_features + nominal_features).to_numpy()), train_df_ordinal_scaled['SalePriceTransformed'].to_numpy())

In [68]:
# best_hp_for_embedding_nn = find_best_hyperparameters_for_nn(train_df_ordinal_scaled)

In [69]:
# embedding_nn = train_nn(train_df_ordinal_scaled, best_hp_for_embedding_nn, shuffle=True)

In [70]:
# nn_predicts = nn_model.predict(get_inp(test_df_ordinal_scaled.filter(all_numerical_features + nominal_features).to_numpy()))

In [71]:
# nn_predicts.max()
#train_df['SalePriceTransformed'].max()

In [72]:
# nn_model.save("./nn_embedding2.keras")

In [73]:
# nn_predicts_real = np.exp(nn_predicts)
# pd.DataFrame(nn_predicts_real).head(20)

In [74]:
# nn_model.save("./nn_embedding1.keras")

In [75]:
#model.summary()
# keras.utils.plot_model(nn_model)

In [76]:
# nn_model.summary()

In [77]:
# with open("./best_nn_hyperparameters5.txt", 'w') as f:
#     f.write(f'{best_hp.values}')